In [1]:
import fitsio, re
from numpy.typing import NDArray, DTypeLike
from typing import Any

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy import integrate
from copy import deepcopy
from astropy.io import fits

import euclidlib as el

from cloelike import EuclidLikelihood_WL_Cls
from cloelib.cosmology.camb_cosmology import CAMBBackground
from cloelib.cosmology.HMcode2020Emu_cosmology import HMemuLinearPerturbations, HMemuNonLinearPerturbations

/Users/pezzotta/miniforge3/envs/cloe-env/lib/python3.11/site-packages/HMcode2020Emu
Loading linear emulator...
Linear emulator loaded in memory.
Loading nonlinear emulator...
Non-linear emulator loaded in memory.
Loading linear emulator...
Baryonic boost emulator loaded in memory.
Loading sigma8 emulator...
Linear emulator loaded in memory.


In [2]:
# Get n(z)
z_nz, nz_heracles = el.photo.redshift_distributions('../playground/tutorials/observables/nz_example.fits')

# Normalise dndz
dndz_pos_norm = deepcopy(nz_heracles)
for key in nz_heracles.keys():
    dndz_pos_norm[key] = nz_heracles[key] / integrate.trapezoid(
                        nz_heracles[key], z_nz)

dndz_she_norm = deepcopy(nz_heracles)
for key in nz_heracles.keys():
    dndz_she_norm[key] = nz_heracles[key] / integrate.trapezoid(
                        nz_heracles[key], z_nz)


# Resampling the normalized n(z) with 100 values
my_dndz_pos = np.vstack(list(dndz_pos_norm.values()))
myz = np.linspace(1e-4, 3., 100)
my_dndz_pos_norm = np.zeros([len(nz_heracles.keys()), 100])
for i in range(6):
    my_dndz_pos_norm[i,:] = np.interp(myz, z_nz, my_dndz_pos[i,:])

my_dndz_she = np.vstack(list(dndz_she_norm.values()))
myz = np.linspace(1e-4, 3., 100)
my_dndz_she_norm = np.zeros([len(nz_heracles.keys()), 100])
for i in range(6):
    my_dndz_she_norm[i,:] = np.interp(myz, z_nz, my_dndz_she[i,:])

In [3]:
cells_data = el.photo.angular_power_spectra('../playground/tutorials/observables/cls_example.fits')
mixmat = el.photo.mixing_matrices('../playground/tutorials/observables/mixmats_example.fits')
full_cov=np.load('../playground/tutorials/observables/cov_Gauss_3x2pt_2D_probe_zpair_ell_2500deg2_Bmode_copy.npy')
full_cov = full_cov[:(21+21)*32, :(21+21)*32]

In [4]:
data = {
    'cells':cells_data,
    'ells':cells_data['SHE', 'SHE', 1, 1].ell,
    'z_arr':myz,
    'dndz_pos':my_dndz_pos_norm,
    'dndz_she':my_dndz_she_norm,
    'cov':full_cov,
    'mixmat':mixmat}

# Scale cuts have the same format as the data
scale_cut_dict = dict.fromkeys(cells_data.keys(), [10,1500])

for key in cells_data.keys():
    if key[:2]==('SHE','SHE'):
        scale_cut_dict[key]=[scale_cut_dict[key],[0,0]]
    
settings = {
    'n_ell_bins': 32,
    'scale_cuts': scale_cut_dict}  

In [5]:
like_test = EuclidLikelihood_WL_Cls.EuclidLikelihoodWLCls(
    data=data, settings=settings,
    Background=CAMBBackground,
    LinPerturbations=HMemuLinearPerturbations,
    NonLinPerturbations=HMemuNonLinearPerturbations)

In [6]:
default_pars = {'H0':67,'Omega_cdm0':0.25,'Omega_b0':0.05,'ns':0.965,'As':2.1e-9,
                'w0':-1,'wa':0, 'Omega_k0':0,'mnu':0.06,'gamma_MG':0.545,
                'log10TAGN': 7.8,
                'AIA':1.72, 'EtaIA':-0.41,
                'multiplicative_bias_1': 0.0, 'multiplicative_bias_2': 0.0,
                'multiplicative_bias_3': 0.0, 'multiplicative_bias_4': 0.0,
                'multiplicative_bias_5': 0.0, 'multiplicative_bias_6': 0.0,
                'dz_shear_1': 0.0, 'dz_shear_2': 0.0,
                'dz_shear_3': 0.0, 'dz_shear_4': 0.0,
                'dz_shear_5': 0.0, 'dz_shear_6': 0.0}

In [7]:
%time like_test.loglike(default_pars)

CPU times: user 12.3 s, sys: 1.38 s, total: 13.7 s
Wall time: 10.6 s


-10.120197623658637